# Titanic：データを読み、仮説を分析する（70分）
**氏名：　　　　学籍番号：　　　　実施日：　　　　**

今回はデータの確認・集計・前処理を重視し、仮説に基づく重回帰と深さ2の決定木を実行します。
モデルの精度比較、データの追加分割、Kaggleへの提出は次回です。

|時間|作業|
|---|---|
|0–5分|仮説|
|5–15分|データの構造|
|15–30分|集計・可視化|
|30–45分|分析の準備|
|45–58分|仮説に基づく重回帰|
|58–67分|決定木|
|67–70分|振り返り・提出|

**操作**：コードセルはShift+Enterで上から実行。回答欄はダブルクリックして編集し、Shift+Enterで確定。
コードを変更したら、そのセルとそれ以降の関連セルを実行し直します。
事前にREADMEの環境準備を済ませ、右上のカーネルでこのフォルダの`.venv`を選んでください。


## 1．仮説を立てる（0–5分）
乗客のどのような特徴が生存と関係しそうでしょうか。事故当時の状況を想像して書きましょう。
ただし、データに含まれない事情も多く、予測に役立つ関連と因果関係は区別します。

**回答1（結果を見る前に記入）**
- 私の仮説：
- その理由：
- 関係しそうな変数：
- 予想する方向（プラス／マイナス）：


## 2．データの構造を確認する（5–15分）
最初のセルは準備用です。`sys.executable`が、この教材の`.venv`内を指すことを確認します。
グラフ内のラベルは、環境による日本語の文字化けを避けるため英語にしています。


In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from IPython.display import display

print('Python:', sys.executable)
pd.set_option('display.max_columns', 20)
plt.rcParams['figure.figsize'] = (7, 4)
# VS Codeでは、titanic70フォルダを「フォルダーを開く」で開いてください。
DATA_DIR = Path('data')
if not (DATA_DIR / 'train.csv').exists():
    raise FileNotFoundError('data/train.csv がありません。READMEの配置と作業フォルダを確認してください。')
train = pd.read_csv(DATA_DIR / 'train.csv')
test = pd.read_csv(DATA_DIR / 'test.csv')
required = {'Survived','Sex','Age','Pclass','Fare','SibSp','Parch','PassengerId'}
assert required <= set(train.columns), 'Kaggle Titanicのtrain.csvを使ってください。'
assert set(train['Survived'].dropna().unique()) <= {0,1}


In [ ]:
display(train.head())
print('train（行数, 列数）:', train.shape)
print('test （行数, 列数）:', test.shape)
print('trainだけにある列:', sorted(set(train.columns) - set(test.columns)))
train.info()
display(train.describe().round(2))


### 変数の辞書
|変数|意味|扱い方の注意|
|---|---|---|
|PassengerId|乗客の識別番号|数量の大小ではない。説明変数に使わない|
|Survived|死亡0／生存1|目的変数。平均は生存率|
|Pclass|1等・2等・3等|回帰ではダミー変数にする|
|Name|氏名|今回は使わない|
|Sex|記録上の性別|femaleを1、maleを0にする|
|Age|年齢|欠損あり|
|SibSp|同乗する兄弟姉妹・配偶者数|自分自身を含まない|
|Parch|同乗する親・子ども数|友人などは含まない|
|Ticket|チケット番号|今回は使わない|
|Fare|運賃|所得そのものではない。共有チケット等にも注意|
|Cabin|客室番号|欠損が多い。今回は使わない|
|Embarked|乗船港 C/Q/S|今回は使わない|

**回答2**
1. trainの乗客数と列数：
2. testに正解の列はありますか：
3. PassengerIdとAgeは、どちらも数値です。意味の違いは：
4. Survivedの平均は何を表しますか：


## 3．集計・可視化（15–30分）
### 3.1 全体の生存者数と割合
人数（count）と割合（share）を区別しましょう。


In [ ]:
counts = train['Survived'].value_counts().sort_index()
display(pd.DataFrame({'人数': counts, '割合': counts / len(train)}))
print(f"全体の生存率: {train['Survived'].mean():.1%}")


### 3.2 性別と旅客クラス
`groupby`でグループを作り、`agg`で人数と平均をまとめます。
生存率だけでなく、人数も確認してください。


In [ ]:
by_sex = train.groupby('Sex')['Survived'].agg(n='size', survival_rate='mean')
by_class = train.groupby('Pclass')['Survived'].agg(n='size', survival_rate='mean')
display(by_sex.round(3))
display(by_class.round(3))
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
by_sex['survival_rate'].plot.bar(ax=axes[0], rot=0, color='#237A8A')
by_class['survival_rate'].plot.bar(ax=axes[1], rot=0, color='#D78B3B')
for ax in axes:
    ax.set_ylim(0, 1)
    ax.set_ylabel('Survival rate')
plt.tight_layout()
plt.show()


### 3.3 性別と旅客クラスを同時に見る
性別の違いは、同じ旅客クラスの中でも見られるでしょうか。


In [ ]:
joint = train.groupby(['Pclass','Sex'])['Survived'].agg(n='size', survival_rate='mean')
display(joint.round(3))
display(train.pivot_table(index='Pclass', columns='Sex', values='Survived', aggfunc='mean').round(3))


### 3.4 年齢分布と欠損
ヒストグラムは各年齢層の人数を示します。年齢不明の乗客はこのグラフに含まれません。


In [ ]:
train['Age'].hist(bins=range(0, 86, 5), color='#237A8A', edgecolor='white')
plt.xlabel('Age (years)')
plt.ylabel('Passengers with observed age')
plt.show()
print('年齢が分かる人数:', train['Age'].notna().sum())
print('年齢不明の人数:', train['Age'].isna().sum())


**回答3**
- 仮説と一致した点（人数・割合などの数値を1つ以上使う）：
- 意外だった点：
- 性別と旅客クラスを同時に見ると分かったこと：

余裕があれば、運賃や同乗家族数と生存の関係も集計してください。


## 4．分析の準備（30–45分）
### 4.1 欠損値は0ではない
欠損率を確認します。欠損の多い列があるという理由だけで、全列について空欄のある行を一括削除しないでください。
元データ`train`は残し、`work`に分析用のコピーを作ります。


In [ ]:
missing = pd.DataFrame({'欠損数': train.isna().sum(), '欠損率': train.isna().mean()})
display(missing.sort_values('欠損数', ascending=False))
work = train.copy()
age_median = train['Age'].median()
fare_median = train['Fare'].median()
work['AgeFilled'] = train['Age'].fillna(age_median)
work['FareFilled'] = train['Fare'].fillna(fare_median)
print('補完に使う年齢の中央値:', age_median)
print('補完に使う運賃の中央値:', fare_median)
# 変更された行を見て、0への置換とは異なることを確認
mask = train['Age'].isna()
display(work.loc[mask, ['PassengerId','Age','AgeFilled']].head())


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, series, title in zip(axes, [train['Age'], work['AgeFilled']], ['Before imputation', 'After median imputation']):
    ax.hist(series.dropna(), bins=range(0, 86, 5), color='#237A8A', edgecolor='white')
    ax.set_title(title)
    ax.set_xlabel('Age')
    ax.set_ylabel('Count')
limit = max(ax.get_ylim()[1] for ax in axes)
for ax in axes:
    ax.set_ylim(0, limit)
plt.tight_layout()
plt.show()


中央値で埋めた年齢は、確認できた実年齢ではありません。中央値付近の人数が人工的に増え、結果も補完方法に依存します。

### 4.2 ダミー変数を作る
基準は男性・1等です。切片を入れる回帰では、各カテゴリを全部ダミーにせず基準を1つ省きます。


In [ ]:
assert work['Sex'].isin(['female','male']).all()
assert work['Pclass'].isin([1,2,3]).all()
work['Female'] = work['Sex'].map({'female': 1, 'male': 0})
work['Class2'] = (work['Pclass'] == 2).astype(int)
work['Class3'] = (work['Pclass'] == 3).astype(int)
display(work[['Sex','Female','Pclass','Class2','Class3']].head(10))


**回答4（コードの出力も使って確認）**
|属性|Female|Class2|Class3|
|---|---|---|---|
|男性・1等|記入|記入|記入|
|女性・2等|記入|記入|記入|
|女性・3等|記入|記入|記入|

- 欠損年齢を0で埋めると、どんな問題がありますか：
- 中央値補完の前後で、分布はどう変わりましたか：


### 4.3 仮説に対応する変数を準備する
基本変数はFemale・AgeFilled・Class2・Class3。追加候補から原則1つ選びます。

|変数名|意味・1単位|仮説の例|
|---|---|---|
|Fare10|運賃10ポンド分|高い運賃の乗客ほど生存しやすい|
|FamilyCount|同乗家族1人分（本人除く）|家族が多いと避難が難しい|
|Alone|記録上の同乗家族なし＝1|家族なしの乗客は生存しにくい|
|Child|年齢16歳未満＝1|子どもは優先的に救助された可能性|

`Child`の年齢不明者は、補完年齢によって分類されます。真の子ども判定ではない点に注意。
16歳という境界は結果を見る前に固定します。


In [ ]:
work['Fare10'] = work['FareFilled'] / 10
work['FamilyCount'] = work['SibSp'] + work['Parch']
work['Alone'] = (work['FamilyCount'] == 0).astype(int)
CHILD_AGE = 16
work['Child'] = (work['AgeFilled'] < CHILD_AGE).astype(int)
base_features = ['Female','AgeFilled','Class2','Class3']
choices = ['Fare10','FamilyCount','Alone','Child']
display(work[base_features + choices].head())
assert work[base_features + choices].notna().all().all()


## 5．仮説に基づく重回帰（45–58分）
生存1・死亡0を目的変数としてOLS（最小二乗法）で重回帰します。これは線形確率モデルです。
係数×100は予測生存率の「パーセントポイント」の差になります。

**まず記入（3分）**
- 追加する変数：
- 仮説とその変数のつながり：
- 係数の予想符号と理由：

下の`extra_features`を編集します。例はFamilyCountです。仮説に合わせて変更してください。
2つ以上入れる場合は教員と相談してください。FamilyCountと、その内訳SibSp・Parchをすべて同時に入れることはできません。


In [ ]:
# ★学生が変更する場所（例：['Fare10']、['Alone']、['Child']）
extra_features = ['FamilyCount']

assert extra_features and set(extra_features) <= set(choices), '候補から1つ以上選んでください。'
selected_features = base_features + extra_features
assert len(selected_features) == len(set(selected_features)), '同じ変数を重複して指定しないでください。'
y = work['Survived'].astype(float)
X_base = sm.add_constant(work[base_features].astype(float), has_constant='add')
X_selected = sm.add_constant(work[selected_features].astype(float), has_constant='add')
assert np.linalg.matrix_rank(X_selected) == X_selected.shape[1], '説明変数が完全に重複しています。'
# HC3は不均一分散に頑健な標準誤差。今日は係数の解釈に集中します。
base_model = sm.OLS(y, X_base).fit(cov_type='HC3')
selected_model = sm.OLS(y, X_selected).fit(cov_type='HC3')
coef_table = pd.DataFrame({'基本モデル': base_model.params, '自分のモデル': selected_model.params})
coef_table['自分のモデル：係数×100'] = coef_table['自分のモデル'] * 100
display(coef_table.round(4))


### 係数の読み方（5分）
- **Female**：年齢・旅客クラス・追加変数が同じとき、男性に対する女性の差。
- **AgeFilled**：他の説明変数が同じとき、年齢1歳あたりの差。ただし補完年齢を含む。
- **Class2／Class3**：他の説明変数が同じとき、1等に対する差。
- **Fare10**：他の説明変数が同じとき、運賃10ポンドあたりの差。
- **FamilyCount**：他の説明変数が同じとき、同乗家族1人あたりの差。
- **Alone**：他の説明変数が同じとき、同乗家族がいる乗客に対する差。
- **Child**：年齢の直線的な関係に追加される、16歳未満側の差。年齢と子どもダミーは同時に動くので、「同年齢の子どもと大人の比較」とは読めない。

例えばFamilyCountの係数が仮に−0.04なら「性別・年齢・旅客クラスが同じ場合、同乗家族が1人多いと予測生存率は4パーセントポイント低い」。これは架空の例です。
切片は男性・1等・年齢0・追加変数0の予測値であり、全体の生存率ではありません。

**回答5**
1. 追加変数の係数と、その意味（比較対象・単位・調整した変数を明記）：
2. 基本モデルから変化した係数を1つ選び、変化を書く：
3. 結果は最初の仮説と整合的ですか：
4. この結果だけでは判断できないこと：

符号が予想どおりでも仮説の証明にはなりません。係数の変化も因果的なメカニズムを証明しません。


In [ ]:
# 数人の予測を読む。正解率の計算・モデル比較は次回。
fitted = selected_model.predict(X_selected)
preview = work[['PassengerId','Survived'] + selected_features].head().copy()
preview['回帰の予測値'] = fitted.head()
preview['0.5以上なら生存と分類'] = (fitted.head() >= 0.5).astype(int)
display(preview.round(3))
print('予測値の最小・最大:', round(fitted.min(), 3), round(fitted.max(), 3))
print('0〜1の範囲外の件数:', ((fitted < 0) | (fitted > 1)).sum())


重回帰の予測値は0〜1を外れる場合があります。その場合は確率としてそのまま解釈できません。
表示した値は学習に使った乗客への当てはめであり、未知の乗客への予測性能の証拠ではありません。


## 6．決定木を読む（58–67分）
自分が重回帰で選んだ特徴をそのまま使います。深さは2に固定します。
決定木は条件で乗客を分け、最後に到達したグループ（葉）で多数派のクラスを予測します。
表示された割合は、その葉の学習データの生存割合です。


In [ ]:
X_tree = work[selected_features].astype(float)
tree = DecisionTreeClassifier(max_depth=2, random_state=55)
tree.fit(X_tree, y.astype(int))
plt.figure(figsize=(14, 7))
plot_tree(tree, feature_names=selected_features,
          class_names=['Died (0)', 'Survived (1)'],
          impurity=False, filled=True, rounded=True, fontsize=10)
plt.title('Decision tree: max_depth = 2')
plt.tight_layout()
plt.show()
print(export_text(tree, feature_names=selected_features))


図の読み方：条件が真なら左、偽なら右に進みます。`samples`は人数、`value`は死亡・生存の内訳（バージョンによって人数または比率）、`class`は予測クラスです。
例：Female ≤ 0.5は男性の側。AgeFilled ≤ ある値なら、その年齢以下の側です。


In [ ]:
# ★0始まりの行番号で乗客を1人選ぶ
row_number = 0
assert 0 <= row_number < len(work)
one = X_tree.iloc[[row_number]]
display(work.iloc[[row_number]][['PassengerId','Survived'] + selected_features])
path_nodes = tree.decision_path(one).indices
for node in path_nodes:
    feature_id = tree.tree_.feature[node]
    if feature_id >= 0:
        feature = selected_features[feature_id]
        threshold = tree.tree_.threshold[node]
        value = one.iloc[0][feature]
        direction = '左' if value <= threshold else '右'
        print(f'{feature} = {value:g}、条件 ≤ {threshold:.3f} → {direction}')
print('予測クラス:', int(tree.predict(one)[0]))
print('到達した葉の生存割合:', round(tree.predict_proba(one)[0, list(tree.classes_).index(1)], 3))


**回答6**
- 最初に分岐した変数と条件：
- 選んだ乗客のPassengerId：
- どの条件を通り、最後に何と予測されましたか：

木に自分の追加変数が現れなくても、「その変数は無関係」とは言えません。深さ制限や他の特徴との関連にも依存します。

## 7．振り返り・提出（67–70分）
**回答7**
- データの集計から分かったこと（数値を1つ以上）：
- 前処理で重要だと思ったこと：
- 自分の仮説と、回帰結果から言えること・言えないこと：

回答欄を埋めて保存し、ファイル名を `titanic_学籍番号.ipynb` にして提出してください。
次回はモデルの比較と未知のデータでの評価を扱います。

### 出典・参照
- データ：[Kaggle Titanic](https://www.kaggle.com/competitions/titanic/data)
- [uvとJupyter](https://docs.astral.sh/uv/guides/integration/jupyter/)
- [statsmodels OLS](https://www.statsmodels.org/stable/generated/statsmodels.regression.linear_model.OLS.html)
- [scikit-learn 決定木](https://scikit-learn.org/stable/modules/tree.html)
